# M17 lab solution: Query Transformation & Routing

**Brief.** Query-transformation matrix: run multi-query, RAG-Fusion, HyDE and decomposition on the same 30 questions and record which question types each one helps, which it hurts, and the added latency and cost.

Run with the default environment: `uv run jupyter lab`.

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from helpers import get_embeddings, get_experientiallabs_llm

llm = get_experientiallabs_llm()
embeddings = get_embeddings()
LLM_NAME = llm.model_name
EMBEDDING_NAME = getattr(embeddings, "model", None) or getattr(embeddings, "endpoint", None)
print("LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)

In [ ]:
# The inline "Part V system", reused by the later labs: the handbook split as lab_kit.baseline splits it
# (recursive, 500 characters, 50 overlap), a FAISS vector retriever over those chunks, and an answer
# prompt that cites numbered passages and is allowed to say "I don't know".
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [
    Document(page_content=c, metadata={"source": p["source"], "section": p["title"]})
    for p in handbook
    for c in splitter.split_text(p["text"])
]
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions from Larkspur Cloud customers using only the numbered handbook passages. "
               "Cite the passages you use as [1], [2]. If the passages do not contain the answer, "
               "reply exactly: I don't know."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata.get('source', '?')}) {d.page_content}" for i, d in enumerate(docs, 1))


def answer(question, docs, config=None):
    return (ANSWER_PROMPT | llm).invoke({"context": format_docs(docs), "question": question}, config=config).content


print(len(chunks), "chunks indexed")

In [ ]:
def texts(docs):
    return [d.page_content for d in docs]


def score(results, questions=None):
    """results: question id -> retrieved chunk texts in rank order; scored with lab_kit on the questions with evidence."""
    qs = questions or golden
    return {"hit_rate": round(hit_rate(results, qs), 3), "recall_at_k": round(recall_at_k(results, qs), 3),
            "mrr": round(mrr(results, qs), 3)}

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler

# Prices in USD per million tokens. Fill them from the provider's price page on the day you run the
# lab and note that date in the README; with None the cost column stays empty.
PRICE_IN_PER_1M = None
PRICE_OUT_PER_1M = None


class Meter(BaseCallbackHandler):
    """Counts chat-model calls and tokens for everything run with config={"callbacks": [meter]}."""

    def __init__(self):
        self.calls = self.tokens_in = self.tokens_out = 0

    def on_llm_end(self, response, **kwargs):
        for gens in response.generations:
            for gen in gens:
                self.calls += 1
                usage = getattr(getattr(gen, "message", None), "usage_metadata", None) or {}
                self.tokens_in += usage.get("input_tokens", 0)
                self.tokens_out += usage.get("output_tokens", 0)

    def cost(self):
        if PRICE_IN_PER_1M is None or PRICE_OUT_PER_1M is None:
            return None
        return (self.tokens_in * PRICE_IN_PER_1M + self.tokens_out * PRICE_OUT_PER_1M) / 1e6

## 1. Five retrieval methods, same budget

Every method returns at most 4 chunks, so they compete on the same context budget.

- **Baseline**: the question as written.
- **Multi-query**: `MultiQueryRetriever` writes three rephrasings and takes the union; the first 4 unique chunks are kept.
- **RAG-Fusion**: four LLM-written queries, each retrieving 4, fused by Reciprocal Rank Fusion (`k = 60`).
- **HyDE**: the LLM writes a hypothetical handbook passage that answers the question; that passage is embedded instead of the question.
- **Decomposition**: the LLM splits the question into at most three sub-questions; each retrieves 2 and the lists are interleaved.

In [ ]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from pydantic import BaseModel, Field


class Queries(BaseModel):
    queries: list[str] = Field(description="Search queries for a customer-support handbook.")


QUERIES_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Write {n} different search queries that would find handbook passages answering the question. "
               "Vary the wording; keep each query short."),
    ("human", "{question}"),
])
SUBQ_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Split the question into at most three self-contained sub-questions, one per fact it needs. "
               "If it needs one fact, return the question unchanged as the only item."),
    ("human", "{question}"),
])
HYDE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Write a short passage from a cloud hosting company's customer handbook that answers the question."),
    ("human", "{question}"),
])
query_writer = QUERIES_PROMPT | llm.with_structured_output(Queries)
decomposer = SUBQ_PROMPT | llm.with_structured_output(Queries)
multi_query = MultiQueryRetriever.from_llm(retriever=retriever, llm=llm)


def unique(docs):
    seen, out = set(), []
    for d in docs:
        if d.page_content not in seen:
            seen.add(d.page_content)
            out.append(d)
    return out


def rrf(lists, k=60):
    scores, by_text = {}, {}
    for docs in lists:
        for rank, d in enumerate(docs, 1):
            scores[d.page_content] = scores.get(d.page_content, 0) + 1 / (k + rank)
            by_text[d.page_content] = d
    return [by_text[t] for t in sorted(scores, key=scores.get, reverse=True)]


def baseline_method(q, cfg):
    return retriever.invoke(q, config=cfg)


def multi_query_method(q, cfg):
    return unique(multi_query.invoke(q, config=cfg))[:4]


def rag_fusion_method(q, cfg):
    queries = query_writer.invoke({"n": 4, "question": q}, config=cfg).queries
    return rrf([retriever.invoke(x, config=cfg) for x in queries])[:4]


def hyde_method(q, cfg):
    passage = (HYDE_PROMPT | llm).invoke({"question": q}, config=cfg).content
    return vectorstore.similarity_search_by_vector(embeddings.embed_query(passage), k=4)


def decomposition_method(q, cfg):
    subs = decomposer.invoke({"question": q}, config=cfg).queries[:3]
    lists = [vectorstore.similarity_search(s, k=2) for s in subs]
    interleaved = [d for rank in range(2) for docs in lists for d in docs[rank:rank + 1]]
    return unique(interleaved)[:4]


METHODS = {"baseline": baseline_method, "multi-query": multi_query_method, "rag-fusion": rag_fusion_method,
           "hyde": hyde_method, "decomposition": decomposition_method}

## 2. Run every method on the 30 questions

Per question: whether the top 4 holds an evidence span (hit), the share of spans found (recall), wall-clock seconds, and the LLM calls and tokens the method spent. Out-of-corpus questions have no spans, so they count towards latency and cost only.

In [ ]:
rows = []
for name, method in METHODS.items():
    for g in golden:
        meter = Meter()
        t0 = time.perf_counter()
        docs = method(g["question"], {"callbacks": [meter]})
        seconds = time.perf_counter() - t0
        found = texts(docs)
        row = {"method": name, "id": g["id"], "type": g["type"], "seconds": seconds, "llm_calls": meter.calls,
               "tokens": meter.tokens_in + meter.tokens_out, "usd": meter.cost()}
        if g["evidence_spans"]:
            row["hit"] = any(is_hit(c, g["evidence_spans"]) for c in found)
            row["recall"] = sum(any(is_hit(c, [s]) for c in found) for s in g["evidence_spans"]) / len(g["evidence_spans"])
        rows.append(row)
    print(name, "done")

## 3. Which question types each method helps or hurts

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
recall = df.dropna(subset=["recall"]).pivot_table(index="type", columns="method", values="recall", aggfunc="mean")
delta = recall.sub(recall["baseline"], axis=0).drop(columns="baseline").round(3)
print("mean recall by type:\n", recall.round(3), "\n")
print("change in recall against the baseline (positive helps, negative hurts):\n", delta, "\n")

base = df[df["method"] == "baseline"].set_index("id")["seconds"]
df["added_seconds"] = df.apply(lambda r: r["seconds"] - base[r["id"]], axis=1)
cost = df.groupby("method").agg(added_seconds=("added_seconds", "mean"), llm_calls=("llm_calls", "mean"),
                                tokens=("tokens", "mean"), usd=("usd", "mean")).round(4)
print("per question, averaged over all 30:\n", cost)

## Results

In [ ]:
summary = df.dropna(subset=["hit"]).groupby("method")[["hit", "recall"]].mean().round(3).join(cost)
print("run:", time.strftime("%Y-%m-%d"), "| LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)
print(summary.to_string())

## What this shows

- The type table is the point of the lab: a method that lifts multi-hop recall can leave lookups unchanged or push a direct match out of the top 4. Read the negative cells as carefully as the positive ones.
- Every method here pays for its gain in LLM calls before retrieval starts; the added seconds and tokens per question are that bill, paid on every query.
- HyDE embeds an answer the model wrote without seeing the handbook, so it can pull towards generic hosting text. Check its numeric and multi-hop rows before using it on a private corpus.